# TP6 : traces, métriques, alertes et diagnostic de lag

| Repère | Valeur |
|---|---|
| Moment | Jour 2, 11:30-12:30, après la démo du TP (11:25-11:30) |
| Compétences | C11 Traces log4j2 modifiées à chaud ; C12 Métriques, tableau de bord et alertes ; C13 Diagnostic d'un consumer lag |
| Slides de rappel | S53 (log4j2 à chaud), S54 et S55 (métriques broker et contrôleur), S57 (agent JMX), S58 (Prometheus et Grafana), S59 (alertes), S60 (lag) |
| Version Markdown | `enonce.md`, mêmes étapes en ligne de commande seule |

Mode requis : `extended`. En mode `core`, Prometheus, Grafana et le générateur de charge n'existent pas : ce TP est impossible.

**Mode d'emploi du notebook**

- Exécutez les cellules dans l'ordre avec `Maj+Entrée`. Après un redémarrage du noyau, les variables sont perdues : relancez les trois cellules du début (connexion, fonctions d'aide, outils du TP), puis les cellules « Code fourni » et la cellule `# TODO` 6.2 a (le contrôleur actif). Ne relancez pas une cellule qui arrête un conteneur : elle referait l'action.
- Chaque étape suit le même schéma : une explication, un **exemple en ligne de commande**, une cellule **`# TODO`**, puis une **vérification**. Les commandes qui modifient le cluster sont affichées sans être exécutées (« ne pas lancer en plus du TODO »).
- **Vous n'avez pas de code à écrire.** Dans une cellule `# TODO`, remplacez chaque `...` par une valeur : un **réglage** (un nom de logger, une option de commande...) ou votre **prévision** du résultat (un nombre, un état, `True` ou `False`). Les noms qui commencent par `PREDICTED_` sont des prévisions : décidez-les avant d'exécuter, la vérification les compare au résultat.
- Sous les réponses, le code est fourni : lisez ses commentaires, ne le modifiez pas. Tant qu'il reste un `...`, la ligne `if filled(...):` l'empêche de s'exécuter et affiche « À compléter ».
- Le notebook joint Prometheus et Grafana par leur nom dans le réseau du lab (`PROMETHEUS`, `GRAFANA`). Depuis votre navigateur, ce sont http://localhost:9090 et http://localhost:3000.
- Les étapes 6.7 et 6.8 attendent des alertes : leurs cellules restent 2 à 4 minutes en exécution (`[*]`). Ne les interrompez pas.

In [ ]:
# Connexion au lab : adresses des brokers, configuration du client Python, dossier de travail
import os
import time

from confluent_kafka.admin import AdminClient

BS = os.environ["BS"]  # listener INTERNAL, comme $BS dans le conteneur tools
PROMETHEUS = "http://prometheus:9090"
CONF = {"bootstrap.servers": BS, "log_level": 2}

# Les commandes docker compose et les fichiers config/ se lisent depuis le dossier lab/
LAB_DIR = os.environ["FORMATION_DIR"] + "/lab"
os.chdir(LAB_DIR)
os.environ["COMPOSE_PROGRESS"] = "plain"  # docker compose sans animation dans le notebook

In [ ]:
# Trois fonctions d'aide, écrites ici : elles ne font pas partie de confluent-kafka
def wait_for(condition, timeout=60, step=2, message="condition"):
    """Rappelle condition() jusqu'à obtenir une valeur vraie, et la renvoie."""
    start = time.time()
    while True:
        try:
            value = condition()
        except Exception:
            value = None
        if value:
            return value
        if time.time() - start >= timeout:
            raise TimeoutError("%s : non atteinte en %d s" % (message, timeout))
        time.sleep(step)

def table(lines, headers):
    """Affiche un tableau HTML dans le notebook."""
    from html import escape
    from IPython.display import HTML, display

    def cell(value):
        return escape(str(value))

    header = "".join(f"<th>{cell(h)}</th>" for h in headers)
    rows = "".join(
        "<tr>"
        + "".join(
            f'<td style="text-align:left; padding:6px">{cell(value)}</td>'
            for value in line
        )
        + "</tr>"
        for line in lines
    )

    display(
        HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{rows}</tbody></table>")
    )

def filled(*answers):
    """Vrai si toutes les réponses d'une cellule TODO sont données ; sinon affiche « À compléter »."""
    for answer in answers:
        if answer is ...:
            print("À compléter : remplacez chaque ... de la cellule par votre réponse, puis exécutez-la de nouveau.")
            return False
    return True

In [ ]:
# Outils du TP : imports, adresse de Grafana, et contrôle du mode du lab
import os
import re
import shutil
import time

import requests
from confluent_kafka import ConsumerGroupTopicPartitions, KafkaException, TopicPartition
from confluent_kafka.admin import NewTopic, OffsetSpec

GRAFANA = "http://grafana:3000"  # nom du conteneur dans le réseau du lab ; http://localhost:3000 depuis le navigateur

# Prometheus n'existe qu'en mode extended : ce TP en dépend
try:
    requests.get(PROMETHEUS + "/-/ready", timeout=3)
    print("Prometheus joignable : mode extended, TP possible")
except requests.RequestException:
    print("Prometheus absent : démarrez le lab en mode extended (scripts/start.sh extended)")

## 1. Objectifs

À la fin du TP, vous savez :

- retrouver où chaque famille de traces est écrite et avec quel niveau ;
- passer un logger en DEBUG sur un seul nœud, sans redémarrage, puis revenir au niveau normal ;
- lire les métriques brutes exposées par l'agent JMX d'un broker ;
- interroger Prometheus en PromQL et interpréter les indicateurs de santé ;
- relier le tableau de bord Grafana aux requêtes qui l'alimentent ;
- faire le ménage dans les consumer groups inactifs ;
- provoquer un lag, le suivre et voir une alerte passer de `pending` à `firing` ;
- écrire, valider et charger une règle d'alerte.

## 2. Prérequis

- Lab démarré en mode `extended` (supervision et générateur de charge) : `scripts/start.sh extended` puis `scripts/validate.sh`.
- TP du jour 1 et TP5 terminés. Les groupes `billing` et `billing-secure` existent.
- Navigateur sur l'hôte : Prometheus sur http://localhost:9090, Grafana sur http://localhost:3000.
- Notions : slides S52 à S60.

## 3. Architecture utilisée

```
  kafka-1..3 (JVM)                              loadgen-producer   loadgen-consumer
  MBeans -> agent JMX :9404/metrics             agent JMX client :9404/metrics
          |                                              |
          |            kafka-exporter :9308 (lag des groupes, lu dans le cluster)
          |                    |                         |
          +--------------------+-------------------------+
                               |  collecte toutes les 15 s
                               v
                Prometheus :9090  --  règles : monitoring/prometheus/rules/*.yml
                               |
                               v
                Grafana :3000  --  tableau de bord « Kafka - Cluster NovaShop (lab) »

  Traces : config/kafka/common/log4j2.yaml (monté sur chaque nœud)
           -> /opt/kafka/logs/server.log, controller.log, state-change.log,
              kafka-authorizer.log, log-cleaner.log, kafka-request.log
           niveau modifiable à chaud : kafka-configs.sh --entity-type broker-loggers
```

| Composant | Rôle dans le TP |
|---|---|
| `loadgen-producer` | Écrit 100 messages/s dans le topic `loadgen` |
| `loadgen-consumer` | Lit `loadgen` avec le groupe `loadgen-consumers` |
| `kafka-exporter` | Publie `kafka_consumergroup_lag`, calculé par le cluster |
| Règle `KafkaConsumerGroupLagHigh` | Lag d'un groupe sur un topic supérieur à 5000 pendant 1 minute (seuil de lab) |

| Lab | Production |
|---|---|
| Seuil de lag fixe de 5000 messages, atteint en moins d'une minute | Seuil exprimé en retard (secondes) ou en tendance, adapté à chaque application |
| Le fichier log4j2 est relu toutes les 30 s (`monitorInterval`, ajout de la formation) | Selon la politique de l'équipe : une relecture automatique permet aussi une erreur de saisie appliquée à chaud |

## 4. Contexte métier

L'équipe Facturation de NovaShop signale des factures émises avec plusieurs minutes de retard. L'exploitation dispose d'une chaîne de supervision complète, mais personne ne l'a encore prise en main. Votre mission : savoir augmenter les traces du contrôleur pendant un incident, retrouver les bons indicateurs, et prouver qu'une alerte prévient l'astreinte quand un consommateur s'arrête.

## 5. Énoncé

1. Identifiez les fichiers de traces et le logger du contrôleur.
2. Passez ce logger en DEBUG sur le seul contrôleur actif, observez l'effet, puis revenez au niveau normal.
3. Retrouvez dans les métriques brutes, puis dans Prometheus et Grafana, les indicateurs de santé du cluster.
4. Supprimez les consumer groups qui ne servent plus.
5. Arrêtez le consommateur du générateur de charge, suivez le lag et l'alerte, puis rétablissez le service.
6. Écrivez une alerte qui détecte l'arrêt du producteur, prouvez qu'elle fonctionne, puis retirez-la.

## 6. Étapes

### 6.1 Où vont les traces

**Exemples en ligne de commande** : la configuration des traces, puis les fichiers d'un nœud.

In [ ]:
!grep -nE 'monitorInterval|fileName' config/kafka/common/log4j2.yaml
!grep -n -A5 'name: org.apache.kafka.controller' config/kafka/common/log4j2.yaml
!docker compose exec -T kafka-1 ls /opt/kafka/logs </dev/null

In [ ]:
# TODO 6.1 : remplacez les deux ... puis exécutez
INTERVAL = ...        # tous les combien de secondes Kafka relit-il ce fichier ? (un nombre, lu ci-dessus)
CONTROLLER_LOG = ...  # dans quel fichier vont les traces du logger org.apache.kafka.controller ? (un nom de fichier)

# Code fourni, à lire : rien à modifier en dessous
interval = None
log_files = []
if filled(INTERVAL, CONTROLLER_LOG):
    # Le fichier de configuration des traces est lisible depuis le notebook
    with open(os.path.join(LAB_DIR, "config/kafka/common/log4j2.yaml")) as f:
        log4j2_text = f.read()
    # monitorInterval: 30  ->  30
    interval = int(re.search(r"monitorInterval:\s*(\d+)", log4j2_text).group(1))
    # fileName: "${sys:kafka.logs.dir}/server.log"  ->  "server.log", une fois par fichier déclaré
    log_files = re.findall(r'fileName: "\$\{sys:kafka\.logs\.dir\}/([^"]+)"', log4j2_text)

print("relecture toutes les", interval, "s ;", log_files)

In [ ]:
# Vérification 6.1
assert interval == 30, "À revoir : fichier relu toutes les 30 s"
print("OK  fichier relu toutes les 30 s")
assert len(log_files) == 6 and "controller.log" in log_files, "À revoir : 6 fichiers de traces, dont controller.log"
print("OK  6 fichiers de traces, dont controller.log")
assert INTERVAL == interval, "À revoir : réponse %r, monitorInterval vaut %s" % (INTERVAL, interval)
print("OK  réponse juste : relecture toutes les %s s" % interval)
assert CONTROLLER_LOG == "controller.log", "À revoir : réponse %r : l'appender ControllerAppender écrit dans controller.log" % (CONTROLLER_LOG,)
print("OK  réponse juste : controller.log")

**Résultat attendu** :

- `monitorInterval: 30` : le fichier est relu toutes les 30 s ;
- 6 fichiers déclarés : `server.log`, `state-change.log`, `kafka-request.log`, `log-cleaner.log`, `controller.log`, `kafka-authorizer.log` ;
- `org.apache.kafka.controller` : niveau `INFO`, `additivity: false`, appender `ControllerAppender`. Ses lignes vont donc dans `controller.log` uniquement ;
- les fichiers tournent toutes les heures (`controller.log.2026-10-28-15` contient l'heure 15, en UTC) ; on trouve aussi les traces du ramasse-miettes (`kafkaServer-gc.log`).

### 6.2 Changer un niveau de trace à chaud

Le niveau d'un logger se change sans redémarrage, nœud par nœud, avec `kafka-configs.sh --entity-type broker-loggers`. Le client Python n'a pas d'API pour cela (`ConfigResource` ne connaît pas ce type) : dans cette étape, c'est une cellule `!docker exec kafka-tools kafka-configs.sh ...` qui appelle l'outil Java.

**Exemples en ligne de commande** : le contrôleur actif (`LeaderId`), le nombre de loggers configurables sur un nœud et le niveau de deux d'entre eux.

In [ ]:
!docker exec kafka-tools kafka-metadata-quorum.sh --bootstrap-server $BS describe --status | grep LeaderId
!docker exec kafka-tools bash -c "kafka-configs.sh --bootstrap-server $BS --describe --entity-type broker-loggers --entity-name 1 | grep -c '='"
!docker exec kafka-tools bash -c "kafka-configs.sh --bootstrap-server $BS --describe --entity-type broker-loggers --entity-name 1 | grep -E ' (org.apache.kafka.controller|root)='"

La fonction `debug_lines(node)` compte les lignes DEBUG de `controller.log` sur un nœud.

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**), pour le contrôleur actif `$L` :

```bash
# [tools]
kafka-configs.sh --bootstrap-server $BS --alter --entity-type broker-loggers --entity-name $L \
  --add-config org.apache.kafka.controller=DEBUG
kafka-configs.sh --bootstrap-server $BS --alter --entity-type broker-loggers --entity-name $L \
  --add-config kafka.controller=DEBUG
kafka-topics.sh --bootstrap-server $BS --create --topic inventory --partitions 3 --replication-factor 3 \
  --config cleanup.policy=compact
```

In [ ]:
# Code fourni : nombre de lignes DEBUG dans controller.log d'un nœud
def debug_lines(node):
    """Compte les lignes DEBUG de controller.log sur kafka-node (grep -c dans le conteneur)."""
    output = !docker compose exec -T kafka-{node} sh -c 'grep -c DEBUG /opt/kafka/logs/controller.log' </dev/null 2>&1
    output = output.n
    return int(output.split()[-1])  # le dernier mot de la sortie : le compte

In [ ]:
# TODO 6.2 a : remplacez les trois ... puis exécutez
LOGGER = ...                    # nom du logger du contrôleur, lu dans log4j2.yaml à l'étape 6.1
PREDICTED_OLD_NAME_OK = ...     # le nom kafka.controller, courant dans les documentations 3.x,
                                # sera-t-il accepté ? (True ou False)
PREDICTED_CREATION_LEVEL = ...  # la création d'un topic est-elle tracée par le contrôleur en "INFO",
                                # ou seulement en "DEBUG" ?

# Code fourni, à lire : rien à modifier en dessous
leader = None
old_name_refusal = creation_logged = ""
debug_in_10s = None
if filled(LOGGER, PREDICTED_OLD_NAME_OK, PREDICTED_CREATION_LEVEL):
    admin = AdminClient(CONF)
    # 1. Le contrôleur actif : LeaderId dans la sortie de kafka-metadata-quorum.sh
    quorum_output = !docker exec kafka-tools kafka-metadata-quorum.sh --bootstrap-server $BS describe --status 2>&1
    quorum_output = quorum_output.n
    leader = int(re.search(r"LeaderId:\s+(\d+)", quorum_output).group(1))

    # 2. Votre logger en DEBUG, sur ce seul nœud (outil Java : pas d'API Python pour les loggers)
    !docker exec kafka-tools bash -c "kafka-configs.sh --bootstrap-server $BS --alter --entity-type broker-loggers --entity-name {leader} --add-config {LOGGER}=DEBUG"
    # 3. Le même réglage avec le nom des documentations 3.x : la réponse est gardée
    old_name_refusal = !docker exec kafka-tools bash -c "kafka-configs.sh --bootstrap-server $BS --alter --entity-type broker-loggers --entity-name {leader} --add-config kafka.controller=DEBUG" 2>&1
    old_name_refusal = old_name_refusal.n

    # 4. Une action d'administration à tracer : création du topic inventory, s'il n'existe pas encore
    if "inventory" not in admin.list_topics(timeout=10).topics:
        admin.create_topics([NewTopic("inventory", num_partitions=3, replication_factor=3,
                                      config={"cleanup.policy": "compact"})])["inventory"].result()

    # 5. Lignes DEBUG écrites en 10 s dans controller.log
    before = debug_lines(leader)
    time.sleep(10)
    debug_in_10s = debug_lines(leader) - before

    # 6. Les lignes de controller.log qui parlent d'inventory, coupées à 200 caractères
    creation_logged = !docker compose exec -T kafka-{leader} sh -c 'grep inventory /opt/kafka/logs/controller.log' </dev/null | cut -c1-200 2>&1
    creation_logged = creation_logged.n

print("contrôleur actif :", leader, "- lignes DEBUG en 10 s :", debug_in_10s)

In [ ]:
# Vérification 6.2 a
assert leader in (1, 2, 3), "À revoir : contrôleur actif : kafka-%s" % leader
print("OK  contrôleur actif : kafka-%s" % leader)
assert "Invalid broker logger" in old_name_refusal, "À revoir : le nom 3.x kafka.controller est refusé"
print("OK  le nom 3.x kafka.controller est refusé")
assert debug_in_10s is not None and debug_in_10s > 20, "À revoir : %s lignes DEBUG en 10 s" % debug_in_10s
print("OK  %s lignes DEBUG en 10 s" % debug_in_10s)
assert "CreateTopics result" in creation_logged and "inventory" in creation_logged, "À revoir : création d'inventory tracée par le contrôleur"
print("OK  création d'inventory tracée par le contrôleur")
assert PREDICTED_OLD_NAME_OK is False, "À revoir : prévision %r : le nom kafka.controller est refusé" % (PREDICTED_OLD_NAME_OK,)
print("OK  prévision juste : nom 3.x refusé")
creation_level = re.search(r"\] (\w+) .*CreateTopics result", creation_logged)
assert creation_level and PREDICTED_CREATION_LEVEL == creation_level.group(1), "À revoir : prévision %r, création tracée en %s" % (PREDICTED_CREATION_LEVEL, creation_level.group(1) if creation_level else "?")
print("OK  prévision juste : création tracée en %s" % creation_level.group(1))

Une ligne DEBUG typique :

In [ ]:
!docker compose exec -T kafka-{leader} sh -c 'grep DEBUG /opt/kafka/logs/controller.log | tail -3' </dev/null | cut -c1-220

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-configs.sh --bootstrap-server $BS --alter --entity-type broker-loggers --entity-name $L \
  --delete-config org.apache.kafka.controller
```

In [ ]:
# TODO 6.2 b : remplacez les deux ... puis exécutez
REMOVE_OPTION = ...          # option de kafka-configs.sh qui retire un réglage dynamique :
                             # "--delete-config" ou "--add-config" ?
PREDICTED_LEVEL_AFTER = ...  # niveau du logger après le retrait : "DEBUG", "INFO" ou "WARN" ?

# Code fourni, à lire : rien à modifier en dessous
final_level = ""
debug_after = None
if filled(REMOVE_OPTION, PREDICTED_LEVEL_AFTER):
    # Retrait du réglage dynamique sur le contrôleur actif
    !docker exec kafka-tools bash -c "kafka-configs.sh --bootstrap-server $BS --alter --entity-type broker-loggers --entity-name {leader} {REMOVE_OPTION} org.apache.kafka.controller"
    # Niveau actuel du logger, lu dans la description des loggers du nœud
    final_level = !docker exec kafka-tools bash -c "kafka-configs.sh --bootstrap-server $BS --describe --entity-type broker-loggers --entity-name {leader} | grep ' org.apache.kafka.controller='" 2>&1
    final_level = final_level.n

    # Lignes DEBUG écrites en 10 s
    before = debug_lines(leader)
    time.sleep(10)
    debug_after = debug_lines(leader) - before

print(final_level, "- lignes DEBUG en 10 s :", debug_after)

In [ ]:
# Vérification 6.2 b
assert "org.apache.kafka.controller=INFO" in final_level, "À revoir : logger du contrôleur revenu à INFO"
print("OK  logger du contrôleur revenu à INFO")
assert debug_after == 0, "À revoir : plus aucune ligne DEBUG"
print("OK  plus aucune ligne DEBUG")
assert "org.apache.kafka.controller=%s" % PREDICTED_LEVEL_AFTER in final_level, "À revoir : prévision %r : %s" % (PREDICTED_LEVEL_AFTER, final_level.strip())
print("OK  prévision juste : %s" % PREDICTED_LEVEL_AFTER)

**Résultat attendu** :

- environ 200 loggers configurables sur un nœud ; `org.apache.kafka.controller` et `root` en `INFO` ;
- le passage en DEBUG répond `Completed updating config for broker-logger <n>.` ; `kafka.controller` est refusé : `Invalid broker logger(s): kafka.controller` ;
- de quelques dizaines à quelques centaines de lignes DEBUG en 10 s, même sans activité d'administration ;
- la création d'`inventory` apparaît en INFO : `CreateTopics result(s): CreatableTopic(name='inventory', ...): SUCCESS`, puis `Replayed TopicRecord`, `Replayed ConfigRecord` et un `Replayed PartitionRecord` par partition ;
- après `--delete-config`, le logger repasse en `INFO` et le compteur DEBUG ne bouge plus.

### 6.3 Lire les métriques brutes

L'agent JMX de chaque nœud publie ses métriques en texte sur le port 9404, une série par ligne : `nom{etiquettes} valeur`. Les lignes qui commencent par `#` décrivent la métrique (`HELP`, `TYPE`).

**Exemple en ligne de commande** : trois métriques de `kafka-1`.

In [ ]:
!docker exec kafka-tools bash -c "wget -qO- http://kafka-1:9404/metrics | grep -E '^kafka_server_replicamanager_(underreplicatedpartitions|leadercount|partitioncount)'"

**Code fourni** : `metrics(node)` lit la page de l'agent JMX d'un nœud et renvoie un dictionnaire `{série: valeur}`.

In [ ]:
# Code fourni : métriques brutes d'un nœud
def metrics(node):
    """{série: valeur} lus sur l'agent JMX de kafka-node (port 9404)."""
    text = requests.get("http://kafka-%d:9404/metrics" % node, timeout=10).text
    series = {}
    for line in text.splitlines():
        # Les lignes # HELP et # TYPE décrivent les métriques : sautées, comme les lignes vides
        if line and not line.startswith("#"):
            series_name, value = line.rsplit(" ", 1)  # « nom{étiquettes} valeur » : la valeur est le dernier mot
            series[series_name] = float(value)
    return series

In [ ]:
# TODO 6.3 : remplacez les deux ... par vos prévisions, puis exécutez
PREDICTED_URP_1 = ...         # valeur de underreplicatedpartitions sur kafka-1, cluster en bonne santé (un nombre)
PREDICTED_ACTIVE_NODES = ...  # sur combien des 3 nœuds activecontrollercount vaut-il 1 ? (un nombre)

# Code fourni, à lire : rien à modifier en dessous
series_count = None
replication_1 = active_controllers = {}
if filled(PREDICTED_URP_1, PREDICTED_ACTIVE_NODES):
    series_1 = metrics(1)
    series_count = len(series_1)  # nombre de séries exposées par kafka-1
    # Trois métriques de réplication de kafka-1
    replication_1 = {name: series_1["kafka_server_replicamanager_" + name]
                     for name in ("leadercount", "partitioncount", "underreplicatedpartitions")}
    # La même métrique, lue sur chacun des 3 nœuds
    active_controllers = {n: metrics(n)["kafka_controller_kafkacontroller_activecontrollercount"] for n in (1, 2, 3)}

print(series_count, "séries ;", replication_1)
print(active_controllers)

In [ ]:
# Vérification 6.3
assert series_count is not None and series_count > 1000, "À revoir : %s séries exposées par kafka-1" % series_count
print("OK  %s séries exposées par kafka-1" % series_count)
assert replication_1.get("underreplicatedpartitions") == 0.0, "À revoir : aucune partition sous-répliquée sur kafka-1"
print("OK  aucune partition sous-répliquée sur kafka-1")
assert sum(active_controllers.values()) == 1 and active_controllers.get(leader) == 1.0, "À revoir : un seul contrôleur actif, le même qu'en 6.2 (kafka-%s)" % leader
print("OK  un seul contrôleur actif, le même qu'en 6.2 (kafka-%s)" % leader)
assert PREDICTED_URP_1 == replication_1["underreplicatedpartitions"], "À revoir : prévision %r, valeur %s" % (PREDICTED_URP_1, replication_1["underreplicatedpartitions"])
print("OK  prévision juste : %d partition sous-répliquée" % replication_1["underreplicatedpartitions"])
assert PREDICTED_ACTIVE_NODES == sum(active_controllers.values()), "À revoir : prévision %r, %d nœud(s) à 1" % (PREDICTED_ACTIVE_NODES, sum(active_controllers.values()))
print("OK  prévision juste : %d contrôleur actif" % sum(active_controllers.values()))

**Résultat attendu** : quelques milliers de séries par nœud ; `underreplicatedpartitions` à 0 ; `activecontrollercount` vaut 1 sur le contrôleur actif trouvé en 6.2 et 0 sur les deux autres.

### 6.4 Interroger Prometheus

Prometheus collecte ces séries toutes les 15 s et répond aux requêtes PromQL par son API : `GET /api/v1/query?query=...`. La réponse JSON contient `data.result`, une liste d'éléments avec `metric` (les étiquettes) et `value` (`[horodatage, "valeur"]`, la valeur étant une chaîne).

**Exemple en ligne de commande** : une requête, depuis `tools`.

In [ ]:
!docker exec kafka-tools bash -c "wget -qO- --post-data 'query=sum(kafka_controller_kafkacontroller_activecontrollercount)' http://prometheus:9090/api/v1/query; echo"

**Code fourni** : `promql(query)` envoie une requête PromQL à Prometheus et renvoie la liste des `(étiquettes, valeur)`. Elle resservira aux étapes 6.5, 6.7 et 6.9.

In [ ]:
# Code fourni : requête PromQL
def promql(query):
    """Liste de (étiquettes, valeur) : une ligne par série du résultat."""
    response = requests.get(PROMETHEUS + "/api/v1/query", params={"query": query}, timeout=10).json()
    # value vaut [horodatage, "valeur"] : la valeur arrive en texte
    return [(r["metric"], float(r["value"][1])) for r in response["data"]["result"]]

In [ ]:
# TODO 6.4 : remplacez les deux ... puis exécutez
QUERY_A = 'sum by (topic) (kafka_server_brokertopicmetrics_messagesin_total{topic!=""})'
QUERY_B = 'sum by (topic) (rate(kafka_server_brokertopicmetrics_messagesin_total{topic!=""}[1m]))'
RATE_QUERY = ...             # requête du débit par topic, en messages par seconde : QUERY_A ou QUERY_B ?
PREDICTED_DOWN_TARGET = ...  # quelle cible de Prometheus est à 0 (up == 0) :
                             # "kafka-1", "kafka-4" ou "loadgen-consumer" ?

# Code fourni, à lire : rien à modifier en dessous
controllers = urp = None
rate = idle = lags = targets = {}
if filled(RATE_QUERY, PREDICTED_DOWN_TARGET):
    # Une seule série dans le résultat : on garde sa valeur
    controllers = promql("sum(kafka_controller_kafkacontroller_activecontrollercount)")[0][1]
    urp = promql("sum(kafka_server_replicamanager_underreplicatedpartitions)")[0][1]
    # Plusieurs séries : un dictionnaire, avec l'étiquette utile pour clé
    rate = {e["topic"]: v for e, v in promql(RATE_QUERY)}
    idle = {e["instance"]: v for e, v in promql("kafka_server_kafkarequesthandlerpool_requesthandleravgidlepercent")}
    lags = {(e["consumergroup"], e["topic"]): v for e, v in promql("sum by (consumergroup, topic) (kafka_consumergroup_lag)")}
    targets = {(e["job"], e["instance"]): v for e, v in promql("up")}

table([("contrôleurs actifs", controllers), ("partitions sous-répliquées", urp),
       ("débit par topic", {t: round(v, 1) for t, v in rate.items() if v > 0}),
       ("inactivité des threads", {i: round(v, 3) for i, v in idle.items()}),
       ("lag par groupe", lags), ("cibles à 0", [c for c, v in targets.items() if v == 0])], ["Indicateur", "Valeur"])

In [ ]:
# Vérification 6.4
assert controllers == 1 and urp == 0, "À revoir : 1 contrôleur actif, 0 partition sous-répliquée"
print("OK  1 contrôleur actif, 0 partition sous-répliquée")
assert 50 < rate.get("loadgen", 0) < 150, "À revoir : débit de loadgen : %s messages/s" % round(rate.get("loadgen", 0), 1)
print("OK  débit de loadgen : %s messages/s" % round(rate.get("loadgen", 0), 1))
assert len(idle) == 3 and min(idle.values()) > 0.5, "À revoir : threads de requêtes peu occupés"
print("OK  threads de requêtes peu occupés")
assert ("loadgen-consumers", "loadgen") in lags, "À revoir : lag de loadgen-consumers publié par kafka-exporter"
print("OK  lag de loadgen-consumers publié par kafka-exporter")
assert [c for c, v in targets.items() if v == 0] == [("kafka-scale", "kafka-4")], "À revoir : seule la cible kafka-4 est à 0"
print("OK  seule la cible kafka-4 est à 0")
assert [PREDICTED_DOWN_TARGET] == [i for (_, i), v in targets.items() if v == 0], "À revoir : prévision %r : voir la ligne « cibles à 0 »" % (PREDICTED_DOWN_TARGET,)
print("OK  prévision juste : %s à 0" % PREDICTED_DOWN_TARGET)

**Résultat attendu** :

| Indicateur | Valeur attendue |
|---|---|
| Contrôleurs actifs | 1 |
| Partitions sous-répliquées | 0 |
| Débit par topic | `loadgen` vers 100 messages/s, environ 1 pour `__consumer_offsets`, 0 pour les topics inactifs |
| Inactivité des threads | Environ 0,99 sur chaque nœud |
| Lag par groupe | Quelques centaines au plus pour `loadgen-consumers`, valeurs fixes pour les groupes arrêtés |
| `up` | 1 pour toutes les cibles, sauf `kafka-4` (job `kafka-scale`) à 0 |

Dans le navigateur : http://localhost:9090, onglet Graph, requête `sum by (topic) (rate(kafka_server_brokertopicmetrics_messagesin_total{topic!=""}[1m]))`, vue Graph sur 15 minutes.

### 6.5 Lire le tableau de bord Grafana

Ouvrez http://localhost:3000, menu Dashboards, puis « Kafka - Cluster NovaShop (lab) », période « Last 15 minutes ». La consultation est ouverte sans compte.

Un tableau de bord Grafana est un document JSON : chaque panneau porte sa requête PromQL. L'API de Grafana le renvoie : `GET /api/dashboards/uid/kafka-cluster`. Dans `dashboard["panels"]`, les panneaux se suivent dans l'ordre d'affichage ; un panneau de type `row` ouvre une rangée, et la requête d'un panneau est dans `panel["targets"][0]["expr"]`.

**Exemple en ligne de commande** : les titres des panneaux, depuis `tools`.

In [ ]:
!docker exec kafka-tools bash -c 'wget -qO- http://grafana:3000/api/dashboards/uid/kafka-cluster | grep -o '"'"'"title":"[^"]*"'"'"' | head -12'

In [ ]:
# TODO 6.5 : ouvrez le tableau de bord dans le navigateur, remplacez les deux ... puis exécutez
PREDICTED_PANELS = ...  # combien de tuiles compte la rangée « Sante du cluster » ? (comptez-les dans Grafana)
GRAFANA_LEADER = ...    # valeur affichée par la tuile « Leader du quorum (node.id) » (un nombre)

# Code fourni, à lire : rien à modifier en dessous
health = []
if filled(PREDICTED_PANELS, GRAFANA_LEADER):
    # Le tableau de bord en JSON : ses panneaux, dans l'ordre d'affichage
    panels = requests.get(GRAFANA + "/api/dashboards/uid/kafka-cluster", timeout=10).json()["dashboard"]["panels"]
    in_row = False  # vrai entre la rangée « Sante du cluster » et la rangée suivante
    for panel in panels:
        if panel["type"] == "row":
            in_row = panel["title"] == "Sante du cluster"
        elif in_row:
            # La requête du panneau, recalculée par Prometheus : la valeur de la tuile
            health.append((panel["title"], promql(panel["targets"][0]["expr"])[0][1]))

table(health, ["Panneau", "Valeur"])

In [ ]:
# Vérification 6.5
health_values = dict(health)
assert len(health) == 8, "À revoir : 8 panneaux dans la rangée Sante du cluster"
print("OK  8 panneaux dans la rangée Sante du cluster")
assert health_values.get("Noeuds joignables") == 3 and health_values.get("Brokers actifs") == 3, "À revoir : 3 nœuds, 3 brokers actifs"
print("OK  3 nœuds, 3 brokers actifs")
assert health_values.get("Leader du quorum (node.id)") == leader, "À revoir : leader du quorum : kafka-%s, comme en 6.2" % leader
print("OK  leader du quorum : kafka-%s, comme en 6.2" % leader)
assert (all(health_values.get(t) == 0 for t in ("Brokers isoles (fenced)", "Partitions sous-repliquees",
                                                "Sous min ISR", "Partitions offline"))), "À revoir : aucun indicateur de panne"
print("OK  aucun indicateur de panne")
assert PREDICTED_PANELS == len(health), "À revoir : réponse %r, %d tuiles dans la rangée" % (PREDICTED_PANELS, len(health))
print("OK  réponse juste : %d tuiles" % len(health))
assert GRAFANA_LEADER == health_values["Leader du quorum (node.id)"], "À revoir : réponse %r, la tuile affiche %s" % (GRAFANA_LEADER, health_values["Leader du quorum (node.id)"])
print("OK  réponse juste : la tuile affiche %d" % GRAFANA_LEADER)

**Résultat attendu** : les 8 valeurs sont celles des tuiles de la rangée « Sante du cluster » dans le navigateur. Dans les autres rangées, « Messages entrants / s par topic » montre `loadgen` vers 100 et « Leaders par broker » une répartition à peu près égale.

### 6.6 Nettoyer les groupes inactifs

**Exemple en ligne de commande** : les groupes et leur état.

In [ ]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --list --state

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-consumer-groups.sh --bootstrap-server $BS --delete --group loadgen-consumers
kafka-consumer-groups.sh --bootstrap-server $BS --delete --group analytics --group analytics-v2
```

In [ ]:
# TODO 6.6 : remplacez les deux ... par vos prévisions, puis exécutez
PREDICTED_ACTIVE_DELETE_OK = ...  # la suppression de loadgen-consumers, groupe STABLE (un consommateur actif),
                                  # sera-t-elle acceptée ? (True ou False)
PREDICTED_OFFSETS_KEPT = ...      # après la suppression d'un groupe, ses offsets validés sont-ils conservés ?
                                  # (True ou False)

# Code fourni, à lire : rien à modifier en dessous
delete_refusal = None
to_delete = remaining_groups = []
if filled(PREDICTED_ACTIVE_DELETE_OK, PREDICTED_OFFSETS_KEPT):
    admin = AdminClient(CONF)
    # 1. Essai sur le groupe actif : le refus éventuel arrive dans une KafkaException
    try:
        admin.delete_consumer_groups(["loadgen-consumers"])["loadgen-consumers"].result()
    except KafkaException as e:
        delete_refusal = e.args[0].name()

    # 2. Groupes inactifs à supprimer, parmi ceux qui existent : analytics, analytics-v2 et ceux du
    #    consommateur console (console-consumer-...)
    existing = [g.group_id for g in admin.list_consumer_groups().result().valid]
    to_delete = sorted(g for g in existing if g in ("analytics", "analytics-v2") or g.startswith("console-consumer-"))
    if to_delete:
        for future in admin.delete_consumer_groups(to_delete).values():
            future.result()
    # 3. Les groupes restants
    remaining_groups = sorted(g.group_id for g in admin.list_consumer_groups().result().valid)

print("refus :", delete_refusal, "- supprimés :", to_delete, "- restants :", remaining_groups)

In [ ]:
# Vérification 6.6
assert delete_refusal == "NON_EMPTY_GROUP", "À revoir : suppression d'un groupe actif refusée"
print("OK  suppression d'un groupe actif refusée")
assert not [g for g in remaining_groups if g.startswith(("analytics", "console-consumer-"))], "À revoir : groupes inactifs supprimés"
print("OK  groupes inactifs supprimés")
assert {"billing", "billing-secure", "loadgen-consumers"} <= set(remaining_groups), "À revoir : billing, billing-secure et loadgen-consumers conservés"
print("OK  billing, billing-secure et loadgen-consumers conservés")
assert PREDICTED_ACTIVE_DELETE_OK == (delete_refusal is None), "À revoir : prévision %r : suppression refusée (%s)" % (PREDICTED_ACTIVE_DELETE_OK, delete_refusal)
print("OK  prévision juste : suppression refusée")
assert PREDICTED_OFFSETS_KEPT is False, "À revoir : prévision %r : supprimer un groupe efface ses offsets validés" % (PREDICTED_OFFSETS_KEPT,)
print("OK  prévision juste : offsets effacés avec le groupe")

**Résultat attendu** : `loadgen-consumers` est `STABLE` ; sa suppression est refusée (`NON_EMPTY_GROUP`, `GroupNotEmptyException` côté Java). Restent `billing`, `billing-secure` et `loadgen-consumers`.

### 6.7 Provoquer et diagnostiquer un lag

Le **lag** d'un groupe est la somme, sur ses partitions, de l'écart entre la fin de la partition et l'offset validé par le groupe.

**Code fourni** : la fonction `watch(count)` relève, toutes les 15 s, le lag de `loadgen-consumers` et l'état des alertes, avec deux autres fonctions fournies, `group_lag(group)` et `alert_states()`. Les temps sont comptés depuis le lancement de la cellule.

In [ ]:
# Code fourni : lag d'un groupe, alertes de Prometheus, et relevé périodique des deux
def group_lag(group):
    """Somme, sur les partitions du groupe, de (fin de la partition - offset validé)."""
    admin = AdminClient(CONF)
    # Offsets validés par le groupe ; un offset négatif signifie « rien de validé » : ignoré
    future = admin.list_consumer_group_offsets([ConsumerGroupTopicPartitions(group)])[group]
    committed = {(tp.topic, tp.partition): tp.offset for tp in future.result().topic_partitions if tp.offset >= 0}
    # Fin de chacune de ces partitions
    ends = admin.list_offsets({TopicPartition(t, p): OffsetSpec.latest() for t, p in committed})
    return sum(f.result().offset - committed[(tp.topic, tp.partition)] for tp, f in ends.items())


def alert_states():
    """{nom de l'alerte: état}, l'état valant "pending" (délai for en cours) ou "firing"."""
    alerts = requests.get(PROMETHEUS + "/api/v1/alerts", timeout=10).json()["data"]["alerts"]
    return {a["labels"]["alertname"]: a["state"] for a in alerts}


def watch(count, step=15, group="loadgen-consumers"):
    """count relevés, un toutes les step secondes : (temps écoulé, lag, alertes)."""
    start, samples = time.time(), []
    for _ in range(count):
        time.sleep(step)
        sample = (round(time.time() - start), group_lag(group), alert_states())
        samples.append(sample)
        print("t+%3d s  lag=%6d  alertes=%s" % sample)
    return samples


print("lag :", group_lag("loadgen-consumers"), "- alertes :", alert_states())

Équivalent en ligne de commande du `TODO` 6.7 a (**ne pas lancer en plus du TODO**) :

```bash
# [hôte]
docker compose stop loadgen-consumer
# [tools] une fois l'alerte en firing
kafka-consumer-groups.sh --bootstrap-server $BS --describe --group loadgen-consumers
```

**Exécutez la cellule `TODO` 6.7 a une seule fois** : elle dure environ 3 minutes. Décidez vos deux prévisions avant de la lancer.

In [ ]:
# TODO 6.7 a : remplacez les deux ... par vos prévisions, puis exécutez UNE SEULE FOIS (3 minutes environ)
PREDICTED_FIRING_DELAY = ...  # au bout de combien de secondes l'alerte KafkaConsumerGroupLagHigh (lag supérieur
                              # à 5000 pendant 1 min) passera-t-elle en firing ? Le producteur écrit 100 messages/s,
                              # Prometheus collecte et évalue toutes les 15 s. (un nombre)
PREDICTED_GROUP_STATE = ...   # état du groupe loadgen-consumers une fois le consommateur arrêté :
                              # "STABLE", "EMPTY" ou "DEAD" ?

# Code fourni, à lire : rien à modifier en dessous
samples_stopped = []
stopped_state = None
if filled(PREDICTED_FIRING_DELAY, PREDICTED_GROUP_STATE):
    admin = AdminClient(CONF)
    !docker compose stop loadgen-consumer
    samples_stopped = watch(12)  # 12 relevés, un toutes les 15 s : 3 minutes
    # État du groupe et nombre de ses membres, consommateur arrêté
    stopped_group = admin.describe_consumer_groups(["loadgen-consumers"])["loadgen-consumers"].result()
    stopped_state = (stopped_group.state.name, len(stopped_group.members))

print("état du groupe :", stopped_state)

In [ ]:
# Vérification 6.7 a
assert samples_stopped and max(r[1] for r in samples_stopped) > 5000, "À revoir : lag au-delà de 5000"
print("OK  lag au-delà de 5000")
assert any(r[2].get("KafkaConsumerGroupLagHigh") == "firing" for r in samples_stopped), "À revoir : alerte KafkaConsumerGroupLagHigh en firing"
print("OK  alerte KafkaConsumerGroupLagHigh en firing")
assert stopped_state == ("EMPTY", 0), "À revoir : groupe sans membre : %s" % (stopped_state,)
print("OK  groupe sans membre : %s" % (stopped_state,))
firing_at = next(r[0] for r in samples_stopped if r[2].get("KafkaConsumerGroupLagHigh") == "firing")
assert abs(PREDICTED_FIRING_DELAY - firing_at) <= 30, "À revoir : prévision %r s, firing au relevé de %d s" % (PREDICTED_FIRING_DELAY, firing_at)
print("OK  prévision juste à 30 s près : firing au relevé de %d s" % firing_at)
assert PREDICTED_GROUP_STATE == stopped_state[0], "À revoir : prévision %r, état %s" % (PREDICTED_GROUP_STATE, stopped_state[0])
print("OK  prévision juste : %s" % stopped_state[0])

Les métriques du consommateur lui-même ont disparu avec lui : sa cible est à 0 et sa métrique de lag n'existe plus.

In [ ]:
print("up du consommateur :", promql('up{instance="loadgen-consumer"}'))
print("records_lag_max :", promql("count(kafka_consumer_consumer_fetch_manager_metrics_records_lag_max)"))

In [ ]:
# TODO 6.7 b : remplacez le ... par votre prévision, puis exécutez (1 min 30 environ)
PREDICTED_CATCHUP = ...  # une fois le consommateur redémarré, le lag repassera-t-il sous 1000
                         # en moins de 90 s ? (True ou False)

# Code fourni, à lire : rien à modifier en dessous
samples_resumed = []
if filled(PREDICTED_CATCHUP):
    !docker compose start loadgen-consumer
    samples_resumed = watch(6)  # 6 relevés : 90 s

In [ ]:
# Vérification 6.7 b
assert samples_resumed and samples_resumed[-1][1] < 1000, "À revoir : lag rattrapé"
print("OK  lag rattrapé")
assert samples_resumed and "KafkaConsumerGroupLagHigh" not in samples_resumed[-1][2], "À revoir : alerte terminée"
print("OK  alerte terminée")
assert PREDICTED_CATCHUP == (samples_resumed[-1][1] < 1000), "À revoir : prévision %r, lag final %d" % (PREDICTED_CATCHUP, samples_resumed[-1][1])
print("OK  prévision juste : lag final %d" % samples_resumed[-1][1])

**Résultat attendu** :

| Temps après l'arrêt | Lag | Alerte `KafkaConsumerGroupLagHigh` |
|---|---|---|
| 15 s | Quelques centaines à 1500 | Aucune |
| 1 min | Environ 5000 à 6000 | Aucune ou `pending` |
| 2 min à 2 min 30 | Environ 12000 à 15000 | `firing` |

- Le lag augmente d'environ 1500 toutes les 15 s, soit le débit du producteur ;
- pendant l'arrêt, le groupe est `EMPTY`, sans membre ;
- après le redémarrage, le lag retombe sous 1000 en moins d'une minute, et l'alerte disparaît.

### 6.8 Défi : écrire une alerte

L'alerte `KafkaLoadgenNoTraffic` doit se déclencher quand le topic `loadgen` reçoit moins de 10 messages par seconde, tous brokers confondus, pendant 1 minute. Prometheus lit ses règles dans `monitoring/prometheus/rules/`, un dossier du lab que le notebook peut écrire : `RULE_FILE` ci-dessous.

La fonction `watch_alert(name)` relève l'état d'une alerte toutes les 15 s, jusqu'à `firing` (4 minutes au plus).

In [ ]:
# Code fourni : fichier de la règle, modèle de règle, et suivi d'une alerte
RULE_FILE = os.path.join(LAB_DIR, "monitoring/prometheus/rules/tp6-custom.yml")  # dossier lu par Prometheus

# Modèle de règle : les deux valeurs entre < > sont remplacées par vos réponses (TODO 6.8 a)
RULE_TEMPLATE = """groups:
  - name: tp6-custom
    rules:
      - alert: KafkaLoadgenNoTraffic
        expr: <requête PromQL : messages entrants par seconde sur le topic loadgen, tous brokers confondus, inférieurs à 10>
        for: <durée>
        labels:
          severity: warning
        annotations:
          summary: "Le topic loadgen reçoit moins de 10 messages/s depuis 1 minute"
"""


def watch_alert(name, count=16, step=15):
    """Relève l'état de l'alerte name toutes les step secondes, jusqu'à firing (count relevés au plus)."""
    start, samples = time.time(), []
    for _ in range(count):
        time.sleep(step)
        samples.append((round(time.time() - start), alert_states().get(name, "absente")))
        print("t+%3d s  %s : %s" % (samples[-1][0], name, samples[-1][1]))
        if samples[-1][1] == "firing":
            break
    return samples

Équivalent en ligne de commande du `TODO` a (**ne pas lancer en plus du TODO**) :

```bash
# [hôte] après avoir écrit monitoring/prometheus/rules/tp6-custom.yml
docker compose exec prometheus promtool check rules /etc/prometheus/rules/tp6-custom.yml
curl -s -w 'HTTP %{http_code}\n' -X POST localhost:9090/-/reload
```

In [ ]:
# TODO 6.8 a (défi) : remplacez les deux ... puis exécutez (la cellule peut être relancée)
EXPR = ...      # l'expression PromQL de l'alerte, en texte. Les briques : la métrique
                # kafka_server_brokertopicmetrics_messagesin_total, le filtre {topic="loadgen"},
                # rate(...[1m]) pour un débit par seconde, sum(...) pour tous les brokers, puis < 10
DURATION = ...  # la durée for, en texte avec son unité, par exemple "30s" ou "5m"

# Code fourni, à lire : rien à modifier en dessous
check = ""
reload_code = None
loaded_rules = []
if filled(EXPR, DURATION):
    # 1. La règle : le modèle, avec vos deux valeurs à la place des < >
    rule = RULE_TEMPLATE.replace(
        "<requête PromQL : messages entrants par seconde sur le topic loadgen, tous brokers confondus, inférieurs à 10>",
        EXPR).replace("<durée>", DURATION)
    with open(RULE_FILE, "w") as f:
        f.write(rule)

    # 2. Validation par promtool, dans le conteneur Prometheus
    check = !docker compose exec -T prometheus promtool check rules /etc/prometheus/rules/tp6-custom.yml </dev/null 2>&1
    check = check.n
    print(check)
    # 3. Rechargement de Prometheus, puis liste des règles chargées
    reload_code = requests.post(PROMETHEUS + "/-/reload", timeout=10).status_code
    loaded_rules = [r["name"] for g in requests.get(PROMETHEUS + "/api/v1/rules", timeout=10).json()["data"]["groups"]
                    for r in g["rules"]]

print("rechargement :", reload_code, "- KafkaLoadgenNoTraffic chargée :", "KafkaLoadgenNoTraffic" in loaded_rules)

In [ ]:
# Vérification 6.8 a
assert "SUCCESS: 1 rules found" in check, "À revoir : règle valide pour promtool"
print("OK  règle valide pour promtool")
assert reload_code == 200 and "KafkaLoadgenNoTraffic" in loaded_rules, "À revoir : règle chargée par Prometheus"
print("OK  règle chargée par Prometheus")

**Exécutez la cellule `TODO` b une seule fois** : elle dure 2 à 3 minutes.

In [ ]:
# TODO 6.8 b : remplacez le ... par votre prévision, puis exécutez UNE SEULE FOIS (2 à 3 minutes)
PREDICTED_IMMEDIATE_PENDING = ...  # 15 s après l'arrêt du producteur, l'alerte sera-t-elle déjà pending ?
                                   # (True ou False ; pensez à la fenêtre [1m] de rate())

# Code fourni, à lire : rien à modifier en dessous
samples_challenge = []
if filled(PREDICTED_IMMEDIATE_PENDING):
    !docker compose stop loadgen-producer
    samples_challenge = watch_alert("KafkaLoadgenNoTraffic")  # jusqu'à firing, 4 minutes au plus
    !docker compose start loadgen-producer

In [ ]:
# Vérification 6.8 b
assert samples_challenge and samples_challenge[-1][1] == "firing", "À revoir : KafkaLoadgenNoTraffic en firing à t+%s s" % (samples_challenge[-1][0] if samples_challenge else "?")
print("OK  KafkaLoadgenNoTraffic en firing à t+%s s" % (samples_challenge[-1][0] if samples_challenge else "?"))
assert PREDICTED_IMMEDIATE_PENDING == (samples_challenge[0][1] != "absente"), "À revoir : prévision %r : au relevé de %d s, l'alerte est %s" % (PREDICTED_IMMEDIATE_PENDING, samples_challenge[0][0], samples_challenge[0][1])
print("OK  prévision juste : au relevé de %d s, l'alerte est %s" % samples_challenge[0])

**Résultat attendu** : `SUCCESS: 1 rules found`, rechargement `200`. Après l'arrêt du producteur : alerte absente pendant 45 s environ, `pending` vers 1 min, `firing` vers 2 min.

### 6.9 Remettre le lab en état

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [hôte]
rm monitoring/prometheus/rules/tp6-custom.yml
curl -s -w 'HTTP %{http_code}\n' -X POST localhost:9090/-/reload
docker compose ps loadgen-producer loadgen-consumer
```

In [ ]:
# TODO 6.9 : remplacez le ... puis exécutez (jusqu'à 3 minutes)
RELOAD = ...  # une fois le fichier de la règle supprimé, faut-il recharger Prometheus
              # pour que la règle disparaisse ? (True ou False)

# Code fourni, à lire : rien à modifier en dessous
load_clients = ""
if filled(RELOAD):
    if os.path.exists(RULE_FILE):
        os.remove(RULE_FILE)  # suppression du fichier de la règle
    if RELOAD:
        requests.post(PROMETHEUS + "/-/reload", timeout=10)
    # Attente : débit de loadgen revenu au-dessus de 50 messages/s, et plus aucune alerte active
    wait_for(lambda: promql('sum(rate(kafka_server_brokertopicmetrics_messagesin_total{topic="loadgen"}[1m]))')[0][1] > 50
             and not alert_states(), timeout=180, step=10, message="débit revenu et aucune alerte")
    # État des deux clients du générateur de charge
    load_clients = !docker compose ps --format '{{{{.Name}}}} {{{{.State}}}}' loadgen-producer loadgen-consumer 2>&1
    load_clients = load_clients.n

print(load_clients)

In [ ]:
# Vérification 6.9
assert not os.path.exists(RULE_FILE), "À revoir : règle personnalisée retirée"
print("OK  règle personnalisée retirée")
assert load_clients.count("running") == 2, "À revoir : les deux clients du générateur tournent"
print("OK  les deux clients du générateur tournent")
assert not alert_states(), "À revoir : aucune alerte active"
print("OK  aucune alerte active")
output = !docker exec kafka-tools bash -c "kafka-configs.sh --bootstrap-server $BS --describe --entity-type broker-loggers --entity-name {leader}" 2>&1
assert ("org.apache.kafka.controller=INFO" in output.n), "À revoir : logger du contrôleur en INFO"
print("OK  logger du contrôleur en INFO")
rules_now = [r["name"] for g in requests.get(PROMETHEUS + "/api/v1/rules", timeout=10).json()["data"]["groups"] for r in g["rules"]]
assert "KafkaLoadgenNoTraffic" not in rules_now, "À revoir : règle encore chargée : Prometheus ne relit ses règles qu'au rechargement"
print("OK  règle déchargée par Prometheus")

## Bilan de l'exercice

In [ ]:
# Code fourni : bilan de l'exercice, une ligne par contrôle ; « à compléter » si l'étape n'a pas été faite
table([
    ("Contrôleur actif", "kafka-%s" % leader, "6.2"),
    ("Lignes DEBUG en 10 s, puis après retour à INFO", "%s, puis %s" % (debug_in_10s, debug_after), "6.2"),
    ("Séries exposées par kafka-1", series_count, "6.3"),
    ("Débit de loadgen (messages/s)", round(rate.get("loadgen", 0), 1) if rate else "à compléter", "6.4"),
    ("Groupes supprimés", ", ".join(to_delete) or "aucun", "6.6"),
    # Premier relevé où l'alerte de lag est en firing
    ("Premier relevé en firing (lag)", next(("t+%d s" % r[0] for r in samples_stopped
                                             if r[2].get("KafkaConsumerGroupLagHigh") == "firing"), "à compléter"), "6.7"),
    ("KafkaLoadgenNoTraffic en firing", "t+%d s" % samples_challenge[-1][0] if samples_challenge else "à compléter", "6.8"),
], ["Contrôle", "Valeur", "Étape"])

## 7. Questions de réflexion

1. Pourquoi passer le logger en DEBUG sur le seul contrôleur actif ? Que perdez-vous si le leadership du quorum change pendant l'incident ?
2. Le niveau modifié par `broker-loggers` survit-il au redémarrage du nœud ? Et une modification de `log4j2.yaml` ?
3. Que signifie une somme de `activecontrollercount` égale à 0 ? Égale à 2 ?
4. `kafka-4` est à `up=0`. Pourquoi n'est-ce pas une alerte ?
5. `requesthandleravgidlepercent` vaut 0,99. Que signifierait une valeur de 0,2, et quelle alerte se déclencherait ?
6. Le lag dépasse 5000 environ 1 minute après l'arrêt, mais l'alerte ne passe en `firing` qu'une minute plus tard. Quel compromis règle la durée `for` ?
7. Le consommateur arrêté, sa métrique `records_lag_max` disparaît. Pourquoi le lag doit-il être mesuré côté cluster ?
8. Un lag de 5000 messages est-il grave ? Comment l'exprimer en temps de retard ?
9. Quel risque prenez-vous en supprimant un groupe vide qui appartient à une application simplement arrêtée ?